In [0]:
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "Source File Format")
# ADF passes this parameter as "Source File Format" (with spaces)
dbutils.widgets.text("Source File Format", "", "Source File Format (ADF)")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = (dbutils.widgets.get("Source File Format") or dbutils.widgets.get("source_file_format")).strip().lower()

In [0]:
landing_path = f"/Volumes/adb_neeleshcaresync/neeleshlanding/neeleshlanding_volume/{source_system}/{table_name}/{landing_timestamp}/"

# Returns the folder listing, or None if the path (or any parent directory) does not exist
def _list_or_none(path):
    try:
        return dbutils.fs.ls(path)
    except Exception:
        return None

landing_files = _list_or_none(landing_path)

if not landing_files:
    msg = f"No new files received for '{table_name}' at {landing_path}. Exiting without loading."
    print(msg)
    dbutils.notebook.exit(msg)

print(f"Found {len(landing_files)} file(s) in {landing_path}")

In [0]:
# landing_path is defined and validated in the previous cell
if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", "true").option("inferSchema", True).load(landing_path)
else:
    landing_df = spark.read.format("parquet").option("header", "true").option("inferSchema", True).load(landing_path) 

In [0]:
from pyspark.sql import functions as F

updated_df = landing_df.withColumn("landing_stamp", F.lit(landing_timestamp)) \
                       .withColumn("insert_timestamp", F.current_timestamp())

In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_neeleshcaresync.bronze.{table_name}")

In [0]:
%sql
select * from adb_neeleshcaresync.bronze.hospitals

In [0]:
f"/volumes/adb-neeleshcaresync/neeleshlanding/neeleshlanding_volume/{source_system}/{table_name}/{landing_timestamp}/"